# 🗄️ Knowledge Base Setup

This notebook provisions the AWS infrastructure for the Clinical Trial Matching Agent's Knowledge Base:

1. **S3 bucket** — Stores trial documents as JSON
2. **Data ingestion** — Downloads trials from ClinicalTrials.gov and uploads to S3
3. **Bedrock Knowledge Base** — Creates and syncs the KB with OpenSearch Serverless

Run this notebook **once** before using the Knowledge Base features in `clinical_trial_matching_agent.ipynb`.

**Prerequisites:**
- AWS credentials configured (`aws configure`)
- Amazon Bedrock model access enabled for `amazon.titan-embed-text-v2:0`
- Permissions: `s3:*`, `bedrock:*`, `aoss:*`, `iam:*` (or use an admin role for setup)

In [ ]:
# Install dependencies (uncomment if needed)
# !pip install boto3 requests

In [ ]:
import json
import logging
import os
import re
import time
import uuid

import boto3
import requests

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)

# Get account info
sts = boto3.client("sts")
account_id = sts.get_caller_identity()["Account"]
region = boto3.session.Session().region_name

print(f"AWS Account: {account_id}")
print(f"Region:      {region}")

## 1. Configuration

Set names for the resources to create. Change these if you have naming conflicts.

In [ ]:
# Resource naming
PROJECT_PREFIX = "clinical-trial-agent"
BUCKET_NAME = f"{PROJECT_PREFIX}-trials-{account_id}-{region}"
KB_NAME = f"{PROJECT_PREFIX}-kb"
COLLECTION_NAME = f"{PROJECT_PREFIX}-collection"
KB_ROLE_NAME = f"{PROJECT_PREFIX}-kb-role"

# Ingestion settings
CONDITION_GROUPS = [
    "breast cancer",
    "non-small cell lung cancer",
    "type 2 diabetes",
    "heart failure",
    "Alzheimer's disease",
    "amyotrophic lateral sclerosis",
]
MAX_PAGES_PER_CONDITION = 2  # ~100 trials per condition, ~500 total

print(f"S3 Bucket:      {BUCKET_NAME}")
print(f"KB Name:        {KB_NAME}")
print(f"Collection:     {COLLECTION_NAME}")
print(f"Conditions:     {len(CONDITION_GROUPS)} groups")

## 2. Create S3 Bucket

In [ ]:
s3 = boto3.client("s3")

try:
    if region == "us-east-1":
        s3.create_bucket(Bucket=BUCKET_NAME)
    else:
        s3.create_bucket(
            Bucket=BUCKET_NAME,
            CreateBucketConfiguration={"LocationConstraint": region},
        )
    print(f"✅ Created bucket: {BUCKET_NAME}")
except s3.exceptions.BucketAlreadyOwnedByYou:
    print(f"✅ Bucket already exists: {BUCKET_NAME}")
except Exception as exc:
    print(f"⚠️ Bucket creation: {exc}")

# Enable versioning
s3.put_bucket_versioning(Bucket=BUCKET_NAME, VersioningConfiguration={"Status": "Enabled"})

# Block public access
s3.put_public_access_block(
    Bucket=BUCKET_NAME,
    PublicAccessBlockConfiguration={
        "BlockPublicAcls": True,
        "IgnorePublicAcls": True,
        "BlockPublicPolicy": True,
        "RestrictPublicBuckets": True,
    },
)
print("✅ Versioning enabled, public access blocked.")

## 3. Ingest Trials from ClinicalTrials.gov → S3

Downloads trials, transforms them into structured documents, and uploads to S3.

In [ ]:
_CT_API_URL = "https://clinicaltrials.gov/api/v2/studies"


def split_eligibility_criteria(text):
    """Split raw eligibility text into inclusion and exclusion lists."""
    if not text:
        return [], []

    text = text.replace("\r\n", "\n").replace("\r", "\n")

    inc_pattern = re.compile(r"^[^\S\n]*(key\s+)?inclusion\s+criteria\s*:?\s*$", re.IGNORECASE | re.MULTILINE)
    exc_pattern = re.compile(r"^[^\S\n]*(key\s+)?exclusion\s+criteria\s*:?\s*$", re.IGNORECASE | re.MULTILINE)

    inc_match = inc_pattern.search(text)
    exc_match = exc_pattern.search(text)

    if inc_match and exc_match:
        if inc_match.start() < exc_match.start():
            inc_block = text[inc_match.end():exc_match.start()]
            exc_block = text[exc_match.end():]
        else:
            exc_block = text[exc_match.end():inc_match.start()]
            inc_block = text[inc_match.end():]
    elif inc_match:
        inc_block, exc_block = text[inc_match.end():], ""
    elif exc_match:
        inc_block, exc_block = text[:exc_match.start()], text[exc_match.end():]
    else:
        inc_block, exc_block = text, ""

    def extract_items(block):
        if not block or not block.strip():
            return []
        bullet_pat = re.compile(r"^[^\S\n]*(?:[-*•]|\d+[.)\s])", re.MULTILINE)
        items = bullet_pat.split(block) if bullet_pat.search(block) else re.split(r"\n\s*\n", block)
        result = []
        for item in items:
            cleaned = re.sub(r"^[-*•]\s*", "", item.strip())
            cleaned = re.sub(r"^\d+[.)\s]*", "", cleaned)
            cleaned = re.sub(r"\s+", " ", cleaned).strip()
            if cleaned:
                result.append(cleaned)
        return result

    return extract_items(inc_block), extract_items(exc_block)


def transform_study(study):
    """Transform a raw ClinicalTrials.gov study to a Trial Document."""
    protocol = study.get("protocolSection", {})
    id_mod = protocol.get("identificationModule", {})
    status_mod = protocol.get("statusModule", {})
    design_mod = protocol.get("designModule", {})
    cond_mod = protocol.get("conditionsModule", {})
    arms_mod = protocol.get("armsInterventionsModule", {})
    elig_mod = protocol.get("eligibilityModule", {})
    contacts_mod = protocol.get("contactsLocationsModule", {})

    phases = design_mod.get("phases", [])
    interventions = [{"type": iv.get("type", ""), "name": iv.get("name", "")} for iv in arms_mod.get("interventions", [])]
    inclusion, exclusion = split_eligibility_criteria(elig_mod.get("eligibilityCriteria", ""))

    locations = [
        {"facility": l.get("facility", ""), "city": l.get("city", ""), "state": l.get("state", ""), "country": l.get("country", "")}
        for l in contacts_mod.get("locations", [])
    ]

    central = contacts_mod.get("centralContacts", [])
    contact_info = {"name": central[0].get("name", ""), "email": central[0].get("email", ""), "phone": central[0].get("phone", "")} if central else {}

    return {
        "nct_id": id_mod.get("nctId", ""),
        "title": id_mod.get("officialTitle", id_mod.get("briefTitle", "")),
        "phase": phases[0] if phases else "N/A",
        "status": status_mod.get("overallStatus", ""),
        "conditions": cond_mod.get("conditions", []),
        "interventions": interventions,
        "inclusion_criteria": inclusion,
        "exclusion_criteria": exclusion,
        "locations": locations,
        "contact_info": contact_info,
        "study_dates": {
            "start": status_mod.get("startDateStruct", {}).get("date", ""),
            "primary_completion": status_mod.get("primaryCompletionDateStruct", {}).get("date", ""),
        },
    }


print("✅ Transformation functions defined.")

In [ ]:
# Run the ingestion pipeline
s3_client = boto3.client("s3")
total_uploaded = 0
seen_ids = set()

for condition in CONDITION_GROUPS:
    print(f"\n📥 Fetching: {condition}...")
    page_token = None

    for page_num in range(MAX_PAGES_PER_CONDITION):
        params = {"query.cond": condition, "pageSize": 50}
        if page_token:
            params["pageToken"] = page_token

        try:
            resp = requests.get(_CT_API_URL, params=params, timeout=30)
            if not resp.ok:
                print(f"   ⚠️ HTTP {resp.status_code} — skipping")
                break
            data = resp.json()
        except Exception as exc:
            print(f"   ⚠️ Request failed: {exc}")
            break

        studies = data.get("studies", [])
        for study in studies:
            nct_id = study.get("protocolSection", {}).get("identificationModule", {}).get("nctId", "")
            if not nct_id or nct_id in seen_ids:
                continue
            try:
                doc = transform_study(study)
                body = json.dumps(doc, indent=2, ensure_ascii=False)
                s3_client.put_object(
                    Bucket=BUCKET_NAME, Key=f"trials/{nct_id}.json",
                    Body=body.encode("utf-8"), ContentType="application/json",
                )
                seen_ids.add(nct_id)
                total_uploaded += 1
            except Exception as exc:
                logger.warning(f"Failed {nct_id}: {exc}")

        page_token = data.get("nextPageToken")
        if not page_token:
            break

    print(f"   Uploaded so far: {total_uploaded}")

print(f"\n✅ Ingestion complete: {total_uploaded} trials in s3://{BUCKET_NAME}/trials/")

## 4. Create IAM Role for the Knowledge Base

In [ ]:
iam = boto3.client("iam")

trust_policy = json.dumps({
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"Service": "bedrock.amazonaws.com"},
        "Action": "sts:AssumeRole",
        "Condition": {
            "StringEquals": {"aws:SourceAccount": account_id},
        },
    }],
})

try:
    role_response = iam.create_role(
        RoleName=KB_ROLE_NAME,
        AssumeRolePolicyDocument=trust_policy,
        Description="Role for Clinical Trial Agent Knowledge Base",
    )
    role_arn = role_response["Role"]["Arn"]
    print(f"✅ Created role: {role_arn}")
except iam.exceptions.EntityAlreadyExistsException:
    role_arn = f"arn:aws:iam::{account_id}:role/{KB_ROLE_NAME}"
    print(f"✅ Role already exists: {role_arn}")

# Attach inline policy for S3 + Bedrock + OpenSearch access
kb_policy = json.dumps({
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": ["s3:GetObject", "s3:ListBucket"],
            "Resource": [f"arn:aws:s3:::{BUCKET_NAME}", f"arn:aws:s3:::{BUCKET_NAME}/*"],
        },
        {
            "Effect": "Allow",
            "Action": ["bedrock:InvokeModel"],
            "Resource": [f"arn:aws:bedrock:{region}::foundation-model/amazon.titan-embed-text-v2:0"],
        },
        {
            "Effect": "Allow",
            "Action": ["aoss:APIAccessAll"],
            "Resource": ["*"],
        },
    ],
})

iam.put_role_policy(RoleName=KB_ROLE_NAME, PolicyName="KBAccessPolicy", PolicyDocument=kb_policy)
print("✅ Policy attached.")

# Wait for IAM propagation
print("⏳ Waiting 10s for IAM propagation...")
time.sleep(10)

## 5. Create OpenSearch Serverless Collection

In [ ]:
aoss = boto3.client("opensearchserverless")

# Encryption policy (required before collection creation)
enc_policy_name = f"{PROJECT_PREFIX}-enc"
try:
    aoss.create_security_policy(
        name=enc_policy_name,
        type="encryption",
        policy=json.dumps({
            "Rules": [{"Resource": [f"collection/{COLLECTION_NAME}"], "ResourceType": "collection"}],
            "AWSOwnedKey": True,
        }),
    )
    print(f"✅ Encryption policy created: {enc_policy_name}")
except aoss.exceptions.ConflictException:
    print(f"✅ Encryption policy already exists: {enc_policy_name}")

# Network policy
net_policy_name = f"{PROJECT_PREFIX}-net"
try:
    aoss.create_security_policy(
        name=net_policy_name,
        type="network",
        policy=json.dumps([{
            "Rules": [{"Resource": [f"collection/{COLLECTION_NAME}"], "ResourceType": "collection"}],
            "AllowFromPublic": True,
        }]),
    )
    print(f"✅ Network policy created: {net_policy_name}")
except aoss.exceptions.ConflictException:
    print(f"✅ Network policy already exists: {net_policy_name}")

# Data access policy
data_policy_name = f"{PROJECT_PREFIX}-data"
caller_arn = sts.get_caller_identity()["Arn"]
try:
    aoss.create_access_policy(
        name=data_policy_name,
        type="data",
        policy=json.dumps([{
            "Rules": [
                {"Resource": [f"collection/{COLLECTION_NAME}"], "ResourceType": "collection",
                 "Permission": ["aoss:CreateCollectionItems", "aoss:UpdateCollectionItems", "aoss:DescribeCollectionItems"]},
                {"Resource": [f"index/{COLLECTION_NAME}/*"], "ResourceType": "index",
                 "Permission": ["aoss:CreateIndex", "aoss:UpdateIndex", "aoss:DescribeIndex",
                                "aoss:ReadDocument", "aoss:WriteDocument"]},
            ],
            "Principal": [caller_arn, role_arn],
        }]),
    )
    print(f"✅ Data access policy created: {data_policy_name}")
except aoss.exceptions.ConflictException:
    print(f"✅ Data access policy already exists: {data_policy_name}")

In [ ]:
# Create the collection
collection_id = None
collection_endpoint = None

try:
    coll_response = aoss.create_collection(
        name=COLLECTION_NAME,
        type="VECTORSEARCH",
        description="Vector store for clinical trial documents",
    )
    collection_id = coll_response["createCollectionDetail"]["id"]
    print(f"✅ Collection created: {COLLECTION_NAME} (ID: {collection_id})")
except aoss.exceptions.ConflictException:
    # Already exists — look it up
    collections = aoss.list_collections(collectionFilters={"name": COLLECTION_NAME})
    if collections["collectionSummaries"]:
        collection_id = collections["collectionSummaries"][0]["id"]
        print(f"✅ Collection already exists: {COLLECTION_NAME} (ID: {collection_id})")

# Wait for collection to become active
print("⏳ Waiting for collection to become ACTIVE (this can take 2-5 minutes)...")
while True:
    detail = aoss.batch_get_collection(ids=[collection_id])
    status = detail["collectionDetails"][0]["status"]
    if status == "ACTIVE":
        collection_endpoint = detail["collectionDetails"][0]["collectionEndpoint"]
        print(f"✅ Collection ACTIVE. Endpoint: {collection_endpoint}")
        break
    elif status in ("FAILED", "DELETED"):
        raise RuntimeError(f"Collection entered {status} state")
    print(f"   Status: {status}...")
    time.sleep(15)

## 5b. Create Vector Index in OpenSearch Serverless

The Bedrock Knowledge Base requires the vector index to **already exist** in the collection before it can be created. We use the OpenSearch HTTP API (via SigV4 auth) to create it.

In [ ]:
!pip install -q opensearch-py requests-aws4auth

In [ ]:
from opensearchpy import OpenSearch, RequestsHttpConnection
from requests_aws4auth import AWS4Auth

# Build SigV4 auth for OpenSearch Serverless
credentials = boto3.Session().get_credentials()
awsauth = AWS4Auth(
    credentials.access_key,
    credentials.secret_key,
    region,
    "aoss",
    session_token=credentials.token,
)

# Parse the collection endpoint (remove https://)
host = collection_endpoint.replace("https://", "")

os_client = OpenSearch(
    hosts=[{"host": host, "port": 443}],
    http_auth=awsauth,
    use_ssl=True,
    verify_certs=True,
    connection_class=RequestsHttpConnection,
    timeout=60,
)

# Vector index configuration for Bedrock KB
# Titan Text Embeddings v2 produces 1024-dim vectors
INDEX_NAME = "bedrock-knowledge-base-default-index"

index_body = {
    "settings": {
        "index": {
            "knn": True,
        }
    },
    "mappings": {
        "properties": {
            "bedrock-knowledge-base-default-vector": {
                "type": "knn_vector",
                "dimension": 1024,
                "method": {
                    "engine": "faiss",
                    "name": "hnsw",
                    "parameters": {},
                },
            },
            "AMAZON_BEDROCK_TEXT_CHUNK": {
                "type": "text",
            },
            "AMAZON_BEDROCK_METADATA": {
                "type": "text",
                "index": False,
            },
        }
    },
}

# Create the index (idempotent — skip if it already exists)
try:
    if os_client.indices.exists(index=INDEX_NAME):
        print(f"✅ Index already exists: {INDEX_NAME}")
    else:
        os_client.indices.create(index=INDEX_NAME, body=index_body)
        print(f"✅ Vector index created: {INDEX_NAME}")
except Exception as exc:
    # If exists check fails, try creating anyway
    try:
        os_client.indices.create(index=INDEX_NAME, body=index_body)
        print(f"✅ Vector index created: {INDEX_NAME}")
    except Exception as inner_exc:
        if "resource_already_exists_exception" in str(inner_exc).lower() or "already exists" in str(inner_exc).lower():
            print(f"✅ Index already exists: {INDEX_NAME}")
        else:
            raise inner_exc

print(f"\n   Index: {INDEX_NAME}")
print(f"   Vector field: bedrock-knowledge-base-default-vector (1024 dim, HNSW/faiss)")
print(f"   Text field: AMAZON_BEDROCK_TEXT_CHUNK")
print(f"   Metadata field: AMAZON_BEDROCK_METADATA")

## 6. Create the Bedrock Knowledge Base

In [ ]:
bedrock_agent = boto3.client("bedrock-agent")

# Create Knowledge Base
try:
    kb_response = bedrock_agent.create_knowledge_base(
        name=KB_NAME,
        description="Clinical trial documents for the matching agent",
        roleArn=role_arn,
        knowledgeBaseConfiguration={
            "type": "VECTOR",
            "vectorKnowledgeBaseConfiguration": {
                "embeddingModelArn": f"arn:aws:bedrock:{region}::foundation-model/amazon.titan-embed-text-v2:0",
            },
        },
        storageConfiguration={
            "type": "OPENSEARCH_SERVERLESS",
            "opensearchServerlessConfiguration": {
                "collectionArn": f"arn:aws:aoss:{region}:{account_id}:collection/{collection_id}",
                "vectorIndexName": "bedrock-knowledge-base-default-index",
                "fieldMapping": {
                    "vectorField": "bedrock-knowledge-base-default-vector",
                    "textField": "AMAZON_BEDROCK_TEXT_CHUNK",
                    "metadataField": "AMAZON_BEDROCK_METADATA",
                },
            },
        },
    )
    knowledge_base_id = kb_response["knowledgeBase"]["knowledgeBaseId"]
    print(f"✅ Knowledge Base created: {KB_NAME} (ID: {knowledge_base_id})")
except bedrock_agent.exceptions.ConflictException:
    # Already exists — find it
    kbs = bedrock_agent.list_knowledge_bases()
    for kb in kbs["knowledgeBaseSummaries"]:
        if kb["name"] == KB_NAME:
            knowledge_base_id = kb["knowledgeBaseId"]
            break
    print(f"✅ Knowledge Base already exists: {KB_NAME} (ID: {knowledge_base_id})")

print(f"\n📋 KNOWLEDGE_BASE_ID = \"{knowledge_base_id}\"")
print("\n👆 Set this in your agent notebook or as an environment variable.")

## 7. Add S3 Data Source and Sync

In [ ]:
# Create data source pointing to the S3 bucket
try:
    ds_response = bedrock_agent.create_data_source(
        knowledgeBaseId=knowledge_base_id,
        name=f"{PROJECT_PREFIX}-s3-source",
        description="S3 bucket with clinical trial JSON documents",
        dataSourceConfiguration={
            "type": "S3",
            "s3Configuration": {
                "bucketArn": f"arn:aws:s3:::{BUCKET_NAME}",
                "inclusionPrefixes": ["trials/"],
            },
        },
    )
    data_source_id = ds_response["dataSource"]["dataSourceId"]
    print(f"✅ Data source created (ID: {data_source_id})")
except bedrock_agent.exceptions.ConflictException:
    # Already exists — find it
    sources = bedrock_agent.list_data_sources(knowledgeBaseId=knowledge_base_id)
    data_source_id = sources["dataSourceSummaries"][0]["dataSourceId"]
    print(f"✅ Data source already exists (ID: {data_source_id})")

In [ ]:
# Start ingestion job to sync S3 data into the Knowledge Base
print("⏳ Starting Knowledge Base sync (ingestion job)...")

job_response = bedrock_agent.start_ingestion_job(
    knowledgeBaseId=knowledge_base_id,
    dataSourceId=data_source_id,
)
job_id = job_response["ingestionJob"]["ingestionJobId"]
print(f"   Job ID: {job_id}")

# Poll for completion
while True:
    job_detail = bedrock_agent.get_ingestion_job(
        knowledgeBaseId=knowledge_base_id,
        dataSourceId=data_source_id,
        ingestionJobId=job_id,
    )
    status = job_detail["ingestionJob"]["status"]
    if status == "COMPLETE":
        stats = job_detail["ingestionJob"].get("statistics", {})
        print(f"\n✅ Sync complete!")
        print(f"   Documents scanned:  {stats.get('numberOfDocumentsScanned', 'N/A')}")
        print(f"   Documents indexed:  {stats.get('numberOfNewDocumentsIndexed', 0) + stats.get('numberOfModifiedDocumentsIndexed', 0)}")
        print(f"   Documents failed:   {stats.get('numberOfDocumentsFailed', 0)}")
        break
    elif status == "FAILED":
        print(f"\n❌ Sync FAILED")
        print(f"   Failure reasons: {job_detail['ingestionJob'].get('failureReasons', 'unknown')}")
        break
    else:
        print(f"   Status: {status}...")
        time.sleep(15)

## 8. Summary & Next Steps

Copy the Knowledge Base ID below and use it in the main agent notebook.

In [ ]:
print("="*60)
print("🎉 Knowledge Base Setup Complete!")
print("="*60)
print(f"\n  S3 Bucket:          {BUCKET_NAME}")
print(f"  Collection:         {COLLECTION_NAME} ({collection_id})")
print(f"  Knowledge Base ID:  {knowledge_base_id}")
print(f"  IAM Role:           {role_arn}")
print(f"  Trials uploaded:    {total_uploaded}")
print(f"\n📋 To use in the agent notebook, either:")
print(f"   1. Set environment variable: export KNOWLEDGE_BASE_ID=\"{knowledge_base_id}\"")
print(f"   2. Or edit the config cell in clinical_trial_matching_agent.ipynb")
print(f"\n{'='*60}")

## 9. Cleanup (Optional)

Run this cell to delete all resources created by this notebook.

In [ ]:
# ⚠️ UNCOMMENT AND RUN TO DELETE ALL RESOURCES
# This is irreversible!

# print("🗑️ Cleaning up resources...")

# # Delete data source
# try:
#     bedrock_agent.delete_data_source(knowledgeBaseId=knowledge_base_id, dataSourceId=data_source_id)
#     print("  Deleted data source")
# except Exception as e:
#     print(f"  Data source: {e}")

# # Delete Knowledge Base
# try:
#     bedrock_agent.delete_knowledge_base(knowledgeBaseId=knowledge_base_id)
#     print("  Deleted Knowledge Base")
# except Exception as e:
#     print(f"  Knowledge Base: {e}")

# # Delete OpenSearch collection
# try:
#     aoss.delete_collection(id=collection_id)
#     print("  Deleted OpenSearch collection")
# except Exception as e:
#     print(f"  Collection: {e}")

# # Delete security/access policies
# for name, ptype in [(enc_policy_name, "encryption"), (net_policy_name, "network")]:
#     try:
#         aoss.delete_security_policy(name=name, type=ptype)
#     except Exception:
#         pass
# try:
#     aoss.delete_access_policy(name=data_policy_name, type="data")
# except Exception:
#     pass
# print("  Deleted OpenSearch policies")

# # Delete S3 objects and bucket
# try:
#     paginator = s3.get_paginator("list_objects_v2")
#     for page in paginator.paginate(Bucket=BUCKET_NAME):
#         objects = [{"Key": obj["Key"]} for obj in page.get("Contents", [])]
#         if objects:
#             s3.delete_objects(Bucket=BUCKET_NAME, Delete={"Objects": objects})
#     s3.delete_bucket(Bucket=BUCKET_NAME)
#     print("  Deleted S3 bucket")
# except Exception as e:
#     print(f"  S3 bucket: {e}")

# # Delete IAM role
# try:
#     iam.delete_role_policy(RoleName=KB_ROLE_NAME, PolicyName="KBAccessPolicy")
#     iam.delete_role(RoleName=KB_ROLE_NAME)
#     print("  Deleted IAM role")
# except Exception as e:
#     print(f"  IAM role: {e}")

# print("\n✅ Cleanup complete.")